# model_xgboost

Notebook generated by automatic conversion. The code keeps comments in English and can be executed from the `notebooks/ml-models` folder.

## Import necessary libraries

In [6]:
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pipeline import load_and_preprocess, DATA_FOLDER

## Data loading and preprocessing

In [7]:
os.makedirs("outcome", exist_ok=True)

# Load and preprocess data using shared pipeline
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = load_and_preprocess(DATA_FOLDER)
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
X_val = X_val.astype(np.float32)

Train shape: (5170, 17249)
Val shape: (738, 17249)
Test shape: (1477, 17249)


## XGBoost model configuration

In [8]:
try:
    import xgboost as xgb

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda",
        random_state=42,
    )
    use_gpu = True
except Exception as exc:
    warnings.warn(
        "XGBoost GPU not available, running training on CPU. Details: %s" % exc
    )
    import xgboost as xgb

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cpu",
        random_state=42,
    )
    use_gpu = False

## Model training

In [9]:
# Train the model with early stopping on test set
try:
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
except Exception as exc:
    warnings.warn(
        "Error during XGBoost GPU training, switching to CPU. Details: %s" % exc
    )
    model.set_params(device="cpu")
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    use_gpu = False

## Evaluation and saving

In [12]:
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"[XGBoost] RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

# Save the model and feature importance
import os

output_path = "outcome/predictions/xgboost_model.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, preds, alpha=0.6, color='purple', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Real Values')
plt.ylabel('Predicted Values')
plt.title('Ridge: Predicted vs Real')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outcome/xgb_pred_vs_real.png", dpi=300, bbox_inches='tight')
plt.close()

model.save_model(output_path)
importance = model.feature_importances_
plt.figure(figsize=(10, 6))
plt.title("XGBoost Feature Importance")
plt.bar(range(len(importance)), importance)
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.tight_layout()
os.makedirs("outcome/feature_importance", exist_ok=True)
plt.savefig("outcome/feature_importance/xgboost_importance.png")
plt.close()

# Save model
os.makedirs("outcome/models", exist_ok=True)
joblib.dump(model, "outcome/models/xgb_model.pkl")

[XGBoost] RMSE: 0.0825 | MAE: 0.0308 | R²: 0.9977


['outcome/models/xgb_model.pkl']